---

# Phase 10 · Section 1 — Bilingual Support (English / Spanish)

### Objective

Make the portfolio accessible in both English and Spanish. A language toggle in the navbar lets users switch languages. The preference is saved in the session so it persists across pages.

This adds a concrete technical signal — internationalisation (i18n) is a real engineering problem — and demonstrates that the portfolio was built with a broader audience in mind.

---

## Prerequisites

* Fully deployed application — from Phase 7
* All public-facing templates in place
* Stable codebase with no outstanding breaking changes

---

## Implementation Steps

1. Install Flask-Babel
2. Configure Babel in the Flask app
3. Wrap all user-facing template strings with `_()`
4. Extract strings and create `.po` translation files
5. Write Spanish translations
6. Compile translations to `.mo` files
7. Add a language switcher route and toggle in the navbar
8. Test both languages across all public pages

---

# 1.1 — Flask-Babel Setup

## Installation

```bash
pip install Flask-Babel
```

Add to `requirements.txt`:

```
Flask-Babel
```

---

## App Configuration

In `app.py`, initialise Babel and define the locale selector:

```python
from flask_babel import Babel

babel = Babel()

def get_locale():
    return session.get('lang', 'en')

def create_app():
    app = Flask(__name__)
    app.config['BABEL_DEFAULT_LOCALE'] = 'en'
    app.config['BABEL_SUPPORTED_LOCALES'] = ['en', 'es']
    babel.init_app(app, locale_selector=get_locale)
    ...
```

---

## babel.cfg

Create `babel.cfg` at the project root. This tells Babel where to find translatable strings:

```ini
[python: app/**.py]
[jinja2: app/templates/**.html]
extensions=jinja2.ext.autoescape,jinja2.ext.with_
```

---

# 1.2 — Wrapping Template Strings

Every user-facing string in a template must be wrapped with `_()` so Babel can extract and translate it.

## Before

```html
<h1>My Projects</h1>
<p>Get in touch with me.</p>
<a href="/about">About</a>
```

## After

```html
<h1>{{ _("My Projects") }}</h1>
<p>{{ _("Get in touch with me.") }}</p>
<a href="/about">{{ _("About") }}</a>
```

Pages to update:

```
templates/base.html          — navbar links, footer
templates/index.html         — hero, featured section headings
templates/projects.html      — page headings, filter labels, sort labels
templates/project_detail.html — section headings (Tech Stack, Problem, Solution, etc.)
templates/about.html         — all copy
templates/contact.html       — form labels, placeholders, button text
templates/errors/404.html    — error message
templates/errors/500.html    — error message
```

---

# 1.3 — Translation Files

## Extract strings

```bash
pybabel extract -F babel.cfg -o messages.pot .
```

This creates `messages.pot` — the template containing all extracted strings.

## Initialise Spanish translation

```bash
pybabel init -i messages.pot -d app/translations -l es
```

This creates `app/translations/es/LC_MESSAGES/messages.po`.

## File structure

```
app/
└── translations/
    └── es/
        └── LC_MESSAGES/
            ├── messages.po     ← editable translation file
            └── messages.mo     ← compiled binary (generated)
```

## Example .po entry

```po
msgid "My Projects"
msgstr "Mis Proyectos"

msgid "Get in touch with me."
msgstr "Ponte en contacto conmigo."

msgid "About"
msgstr "Sobre mí"

msgid "Contact"
msgstr "Contacto"

msgid "Home"
msgstr "Inicio"

msgid "Featured Projects"
msgstr "Proyectos Destacados"

msgid "View all projects"
msgstr "Ver todos los proyectos"

msgid "Tech Stack"
msgstr "Stack Tecnológico"

msgid "Problem"
msgstr "Problema"

msgid "Solution"
msgstr "Solución"

msgid "Results"
msgstr "Resultados"

msgid "Send Message"
msgstr "Enviar Mensaje"
```

## Compile translations

```bash
pybabel compile -d app/translations
```

Run this every time you update the `.po` file.

## Update translations (when new strings are added)

```bash
pybabel extract -F babel.cfg -o messages.pot .
pybabel update -i messages.pot -d app/translations
pybabel compile -d app/translations
```

---

# 1.4 — Language Switcher

## Route

```python
@app.route('/set-language/<lang>')
def set_language(lang):
    if lang in ['en', 'es']:
        session['lang'] = lang
    return redirect(request.referrer or url_for('index'))
```

## Navbar toggle

```html
<div class="nav-item d-flex align-items-center gap-2">
  <a href="{{ url_for('set_language', lang='en') }}"
     class="nav-link {% if session.get('lang', 'en') == 'en' %}fw-bold{% endif %}">
    EN
  </a>
  <span class="text-muted">|</span>
  <a href="{{ url_for('set_language', lang='es') }}"
     class="nav-link {% if session.get('lang') == 'es' %}fw-bold{% endif %}">
    ES
  </a>
</div>
```

---

# Files Involved

```bash
babel.cfg                              # Babel extraction config
messages.pot                           # Extracted strings template
app/translations/es/LC_MESSAGES/messages.po   # Spanish translations
app/translations/es/LC_MESSAGES/messages.mo   # Compiled binary
app/app.py                             # Babel init, locale selector, set_language route
app/templates/base.html                # Language toggle in navbar
app/templates/*.html                   # All user-facing strings wrapped with _()
requirements.txt                       # Flask-Babel added
```

---

# Validation Checklist

* EN toggle shows the site in English
* ES toggle shows the site in Spanish
* Language preference persists across page navigation
* All navbar links translate correctly
* All page headings translate correctly
* Contact form labels and button translate correctly
* Project detail section headings translate correctly
* No untranslated strings visible in Spanish mode
* Admin routes are unaffected (admin is English only)

---

# Result

The portfolio is now bilingual. English and Spanish users can navigate the full public site in their language. The language switcher is visible in the navbar and the preference is remembered throughout the session.

This adds a concrete i18n implementation to the portfolio's proof signals — a feature most junior portfolios do not have.

---
